<a href="https://colab.research.google.com/github/darkknightamkit/AI-agent/blob/main/ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai python-dotenv

In [23]:
import json
import subprocess
from google.colab import userdata
from openai import OpenAI

# Initialize client using Groq endpoint and secrets manager
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
)


# --- TOOL 1: Save Code ---
def save_code_to_file(arguments):
  filename = arguments.get("filename")
  code_content = arguments.get("code_content")
  with open(filename, "w", encoding="utf-8") as f:
    f.write(code_content)
  return json.dumps(
      {"status": "success", "message": f"Successfully saved code to {filename}"}
  )


# --- TOOL 2: Read Code File ---
def read_code_file(arguments):
  filename = arguments.get("filename")
  try:
    with open(filename, "r", encoding="utf-8") as f:
      content = f.read()
    return json.dumps({"status": "success", "file_content": content})
  except Exception as e:
    return json.dumps({"error": str(e)})


# --- TOOL 3: Execute Code ---
def execute_python_file(arguments):
  filename = arguments.get("filename")
  try:
    result = subprocess.run(
        ["python", filename], capture_output=True, text=True, timeout=30
    )
    return json.dumps({
        "exit_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    })
  except Exception as e:
    return json.dumps({"error": str(e)})


# --- TOOL 4: Pip Install (NEW!) ---
def pip_install(arguments):
  package_name = arguments.get("package_name")
  try:
    result = subprocess.run(
        ["pip", "install", package_name],
        capture_output=True,
        text=True,
        timeout=60,
    )
    return json.dumps({
        "exit_code": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    })
  except Exception as e:
    return json.dumps({"error": str(e)})


# Map all four tools to names
available_functions = {
    "save_code_to_file": save_code_to_file,
    "read_code_file": read_code_file,
    "execute_python_file": execute_python_file,
    "pip_install": pip_install,
}

# --- TOOL SCHEMA (Now with 4 tools) ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "save_code_to_file",
            "description": "Saves or overwrites code snippets into a local file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name of the file, e.g. script.py",
                    },
                    "code_content": {
                        "type": "string",
                        "description": "The exact source code to write.",
                    },
                },
                "required": ["filename", "code_content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_code_file",
            "description": (
                "Reads and returns the contents of an existing file on disk."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": (
                            "The name of the file to read, e.g. script.py"
                        ),
                    }
                },
                "required": ["filename"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "execute_python_file",
            "description": "Executes a local Python script and returns output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name of the python file to run.",
                    }
                },
                "required": ["filename"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "pip_install",
            "description": (
                "Installs any external Python package using pip (e.g. pandas,"
                " requests, matplotlib)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "package_name": {
                        "type": "string",
                        "description": (
                            "The name of the Python package to install, e.g."
                            " 'requests' or 'pandas'"
                        ),
                    }
                },
                "required": ["package_name"],
            },
        },
    },
]

print("Cell 1 updated successfully with the new pip_install tool!")


# --- AGENT LOOP ---
def run_coding_agent(user_prompt):
  messages = [
      {
          "role": "system",
          "content": (
              "You are an autonomous coding agent with file reading, writing,"
              " and execution tools. When asked to modify or fix an existing"
              " file, first use read_code_file to inspect it, then make your"
              " edits using save_code_to_file, and verify by running it."
          ),
      },
      {"role": "user", "content": user_prompt},
  ]

  for iteration in range(7):
    print(f"\n--- Step {iteration + 1}: Thinking... ---")

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b", messages=messages, tools=tools
    )

    assistant_message = response.choices[0].message
    messages.append(assistant_message)

    if assistant_message.tool_calls:
      for tool_call in assistant_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        print(
            f"⚡ Agent Action: Calling tool '{function_name}' with arguments"
            f" {function_args}"
        )

        if function_name in available_functions:
          tool_output = available_functions[function_name](function_args)
        else:
          tool_output = json.dumps({"error": "Unknown function"})

        print(f"👁️ Observation Result:\n{tool_output}")

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_output,
        })
    else:
      print("🏁 Agent Finished Task.")
      return assistant_message.content

  return "Reached maximum steps without final response."

In [ ]:
 goal = (
    "Open the existing file 'primes.py' using the read_code_file tool, inspect"
    " its contents, modify the code so it also prints how many prime numbers"
    " were found in total, save the updated code back to 'primes.py', run it"
    " to verify, and show me the output."
)
final_response = run_coding_agent(goal)
print("\nFinal Agent Response:\n", final_response)

In [36]:
from IPython.display import HTML, clear_output, display
import ipywidgets as widgets
import json


def start_interactive_agent_clean_ui():
  print("🤖 Interactive Coding Agent (Clean Text & Split UI) Initialized!\n" + "=" * 50)

  # System prompt refined for clean, natural language, proper punctuation, and distinct line breaks
  messages = [{
      "role": "system",
      "content": (
          "You are an autonomous coding agent with file reading, writing, and"
          " execution tools. Help the user step-by-step, answer questions,"
          " write code, test it, and fix bugs interactively.\n\nIMPORTANT for"
          " your final response: When you present your final solution, you MUST"
          " format your response using these exact markers so it can be split"
          " into a UI:\n---CODE---\n[Your clean Python code implementation"
          " here]\n---EXPLANATION---\n[Provide a clear, natural-language"
          " dry-run walkthrough. Write in smooth, natural sentences with proper"
          " grammar, correct punctuation, commas, and clear paragraph"
          " line-breaks. Break the dry-run down into structured, easy-to-read"
          " numbered steps or bullet points, showing how variables change and"
          " how the code flows line-by-line, without messy blocks of text.]"
      ),
  }]

  # Multi-line text box widget
  input_box = widgets.Textarea(
      placeholder=(
          "Type your prompt or instructions here (you can write multi-line"
          " text)..."
      ),
      description="Prompt:",
      layout=widgets.Layout(width="100%", height="100px"),
  )

  submit_btn = widgets.Button(
      description="Send to Agent",
      button_style="success",
      icon="paper-plane",
  )

  output_area = widgets.Output()

  def on_submit_clicked(b):
    user_input = input_box.value.strip()
    if not user_input:
      return

    if user_input.lower() in ["exit", "quit"]:
      with output_area:
        display(
            HTML(
                "<div"
                ' style="background:#1e1e1e; color:#f48771; padding:12px;'
                ' border-radius:6px; margin:10px 0; font-family:monospace;">👋'
                " Exiting interactive session.</div>"
            )
        )
      input_box.disabled = True
      submit_btn.disabled = True
      return

    with output_area:
      code_style_fix = """
            <style>
                pre, code {
                    background-color: #1e1e1e !important;
                    color: #d4d4d4 !important;
                    font-family: 'Courier New', Courier, monospace !important;
                    padding: 8px 12px !important;
                    border-radius: 6px !important;
                    font-size: 12px !important;
                    overflow-x: auto !important;
                }
            </style>
            """

      # Render User Input
      display(
          HTML(
              code_style_fix
              + f"<div"
              ' style="background: #252526; border-left: 4px solid #007acc;'
              ' color: #d4d4d4; padding: 12px 15px; margin: 12px 0;'
              ' border-radius: 6px; font-family: sans-serif; box-shadow: 0 2px'
              f' 4px rgba(0,0,0,0.3);"><b>👤 You:</b><div'
              f' style="margin-top: 6px; white-space: pre-wrap;">{user_input}</div></div>'
          )
      )

      messages.append({"role": "user", "content": user_input})

      # Run the agent loop for this turn
      for iteration in range(6):
        display(
            HTML(
                f"<div"
                ' style="color: #858585; font-size: 11px; margin: 6px 0;'
                ' font-family: monospace;">--- Thinking (Step'
                f" {iteration + 1})... ---</div>"
            )
        )
        try:
          response = client.chat.completions.create(
              model="openai/gpt-oss-120b", messages=messages, tools=tools
          )

          assistant_message = response.choices[0].message
          messages.append(assistant_message)

          if assistant_message.tool_calls:
            for tool_call in assistant_message.tool_calls:
              function_name = tool_call.function.name
              function_args = json.loads(tool_call.function.arguments)

              # Render Tool Action Card
              display(
                  HTML(
                      f"<div"
                      ' style="background: #2d2d2d; border: 1px solid #3c3c3c;'
                      ' border-left: 4px solid #ce9178; padding: 10px 12px;'
                      ' margin: 8px 0; border-radius: 6px; font-family:'
                      ' monospace; font-size: 12px; color: #d4d4d4;">⚡'
                      " <b>Tool Call:</b> <span"
                      f" style='color: #4ec9b0;'>{function_name}</span><br><pre"
                      f" style='margin: 6px 0 0 0;'>Args: {json.dumps(function_args, indent=2)}</pre></div>"
                  )
              )

              if function_name in available_functions:
                tool_output = available_functions[function_name](function_args)
              else:
                tool_output = json.dumps({"error": "Unknown function"})

              # Render Observation Result Card
              display(
                  HTML(
                      f"<div"
                      ' style="background: #1e1e1e; border: 1px solid #3c3c3c;'
                      ' border-left: 4px solid #569cd6; padding: 10px 12px;'
                      ' margin: 8px 0; border-radius: 6px; font-family:'
                      ' monospace; font-size: 11px; color: #d4d4d4; max-height:'
                      f" 180px; overflow-y: auto;'>👁️"
                      f" <b>Observation:</b><br><pre style='margin: 6px 0 0 0;'>{tool_output}</pre></div>"
                  )
              )

              messages.append({
                  "role": "tool",
                  "tool_call_id": tool_call.id,
                  "content": tool_output,
              })
          else:
            content_text = (
                assistant_message.content
                if assistant_message.content
                else "(Completed actions)"
            )

            # Parse out CODE and EXPLANATION using markers
            code_part = content_text
            explanation_part = "Review the implementation on the left."

            if "---CODE---" in content_text and "---EXPLANATION---" in content_text:
              parts = content_text.split("---EXPLANATION---")
              code_part = parts[0].replace("---CODE---", "").strip()
              explanation_part = parts[1].strip()

            # Render Split View Final Response with Clean Natural Text and Line Spacing
            display(
                HTML(
                    f"""
                    <div style="background: #222222; border: 1px solid #3c3c3c; border-left: 4px solid #4ec9b0; padding: 14px 16px; margin: 12px 0; border-radius: 6px; font-family: sans-serif; color: #d4d4d4; box-shadow: 0 2px 4px rgba(0,0,0,0.3);">
                        <h4 style="margin: 0 0 12px 0; color: #4ec9b0; font-size: 14px;">🤖 Agent Response & Dry-Run Trace</h4>
                        <div style="display: flex; gap: 15px; flex-wrap: wrap;">
                            <!-- Box 1: Solution Code -->
                            <div style="flex: 1; min-width: 280px; background: #1e1e1e; border: 1px solid #333333; padding: 12px; border-radius: 6px;">
                                <h5 style="margin: 0 0 8px 0; color: #ce9178; font-family: monospace; font-size: 13px;">📦 Solution Code</h5>
                                <div style="font-size: 12px; line-height: 1.5; white-space: pre-wrap; font-family: monospace;">{code_part}</div>
                            </div>
                            <!-- Box 2: Clean Natural Explanation -->
                            <div style="flex: 1; min-width: 280px; background: #252526; border: 1px solid #333333; padding: 12px; border-radius: 6px;">
                                <h5 style="margin: 0 0 8px 0; color: #569cd6; font-family: sans-serif; font-size: 13px;">📖 Step-by-Step Dry Run</h5>
                                <div style="font-size: 12px; line-height: 1.6; white-space: pre-wrap;">{explanation_part}</div>
                            </div>
                        </div>
                    </div>
                    """
                )
            )
            break
        except Exception as e:
          display(
              HTML(
                  f"<div"
                  ' style="background:#2d2d2d; color:#f48771; padding:10px;'
                  ' border-radius:6px; margin:8px 0; font-family:monospace;">❌'
                  f" Error: {e}</div>"
              )
          )
          break

    # Clear input box after submission
    input_box.value = ""

  submit_btn.on_click(on_submit_clicked)

  # Layout container
  display(
      widgets.VBox([
          widgets.HTML(
              "<h3 style='margin-bottom:5px; font-family:sans-serif;'>🚀"
              " AI Agent Workspace (Clean Text UI)</h3>"
          ),
          input_box,
          submit_btn,
          output_area,
      ])
  )


# Run the Clean Text Split UI agent!
start_interactive_agent_clean_ui()

🤖 Interactive Coding Agent (Clean Text & Split UI) Initialized!


In [37]:
# Create requirements.txt
requirements_content = """streamlit>=1.30.0
openai>=1.0.0
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
  f.write(requirements_content)

# Create app.py (The Streamlit Web App version of your Agent)
app_content = """
import streamlit as st
import json
import subprocess
from openai import OpenAI

# Page Config
st.set_page_config(page_title="AI Coding Agent Workspace", layout="wide", page_icon="🤖")

# Custom Dark Theme & Split View Styling
st.markdown('''
    <style>
        .stApp { background-color: #1e1e1e; color: #d4d4d4; }
        .code-box { background-color: #1e1e1e; color: #ce9178; padding: 15px; border-radius: 6px; font-family: monospace; border: 1px solid #333; overflow-x: auto; }
        .expl-box { background-color: #252526; color: #d4d4d4; padding: 15px; border-radius: 6px; font-family: sans-serif; border: 1px solid #333; line-height: 1.6; }
        .card-header { font-size: 14px; font-weight: bold; margin-bottom: 8px; }
    </style>
''', unsafe_allow_html=True)

st.title("🚀 Autonomous AI Coding Agent")
st.caption("Your multi-file developer assistant with file tools, execution, and step-by-step dry-run explanations.")

# Initialize OpenAI/Groq Client using Streamlit Secrets for security
api_key = st.secrets.get("GROQ_API_KEY", "")
if not api_key:
    st.error("⚠️ GROQ_API_KEY missing! Please add it to your Streamlit Cloud Secrets.")
    st.stop()

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=api_key)

# Tool definitions
def save_code_to_file(args):
    try:
        with open(args.get("filename"), "w", encoding="utf-8") as f:
            f.write(args.get("code_content"))
        return json.dumps({"status": "success", "message": f"Saved {args.get('filename')}"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def read_code_file(args):
    try:
        with open(args.get("filename"), "r", encoding="utf-8") as f:
            return json.dumps({"status": "success", "file_content": f.read()})
    except Exception as e:
        return json.dumps({"error": str(e)})

def execute_python_file(args):
    try:
        res = subprocess.run(["python", args.get("filename")], capture_output=True, text=True, timeout=30)
        return json.dumps({"exit_code": res.returncode, "stdout": res.stdout, "stderr": res.stderr})
    except Exception as e:
        return json.dumps({"error": str(e)})

def pip_install(args):
    try:
        res = subprocess.run(["pip", "install", args.get("package_name")], capture_output=True, text=True, timeout=60)
        return json.dumps({"exit_code": res.returncode, "stdout": res.stdout, "stderr": res.stderr})
    except Exception as e:
        return json.dumps({"error": str(e)})

available_functions = {
    "save_code_to_file": save_code_to_file,
    "read_code_file": read_code_file,
    "execute_python_file": execute_python_file,
    "pip_install": pip_install
}

tools = [
    {"type": "function", "function": {"name": "save_code_to_file", "description": "Saves code to a file.", "parameters": {"type": "object", "properties": {"filename": {"type": "string"}, "code_content": {"type": "string"}}, "required": ["filename", "code_content"]}}},
    {"type": "function", "function": {"name": "read_code_file", "description": "Reads a code file.", "parameters": {"type": "object", "properties": {"filename": {"type": "string"}}, "required": ["filename"]}}},
    {"type": "function", "function": {"name": "execute_python_file", "description": "Runs a python script.", "parameters": {"type": "object", "properties": {"filename": {"type": "string"}}, "required": ["filename"]}}},
    {"type": "function", "function": {"name": "pip_install", "description": "Installs a package via pip.", "parameters": {"type": "object", "properties": {"package_name": {"type": "string"}}, "required": ["package_name"]}}}
]

# Session state history
if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "system",
        "content": "You are an autonomous coding agent with file reading, writing, execution, and pip tools. Format your final response with exactly: ---CODE--- [code here] ---EXPLANATION--- [clean step-by-step dry-run explanation here]."
    }]

# Display chat history
for msg in st.session_state.messages[1:]:
    if msg["role"] == "user":
        st.markdown(f"**👤 You:** {msg['content']}")
    elif msg["role"] == "assistant" and msg.get("content"):
        content = msg["content"]
        if "---CODE---" in content and "---EXPLANATION---" in content:
            parts = content.split("---EXPLANATION---")
            code_part = parts[0].replace("---CODE---", "").strip()
            expl_part = parts[1].strip()

            col1, col2 = st.columns(2)
            with col1:
                st.markdown('<div class="card-header" style="color: #ce9178;">📦 Solution Code</div>', unsafe_allow_html=True)
                st.code(code_part, language="python")
            with col2:
                st.markdown('<div class="card-header" style="color: #569cd6;">📖 Step-by-Step Dry Run</div>', unsafe_allow_html=True)
                st.markdown(f'<div class="expl-box">{expl_part}</div>', unsafe_allow_html=True)
        else:
            st.info(content)

# User Input Box
user_prompt = st.text_area("Type your prompt or instructions here:", placeholder="e.g., Build a command-line To-Do app across multiple files...")

if st.button("Send to Agent", type="primary"):
    if user_prompt.strip():
        st.session_state.messages.append({"role": "user", "content": user_prompt})

        with st.spinner("🤖 Agent is thinking, writing code, and testing..."):
            for _ in range(6):
                response = client.chat.completions.create(
                    model="openai/gpt-oss-120b",
                    messages=st.session_state.messages,
                    tools=tools
                )
                assistant_message = response.choices[0].message
                st.session_state.messages.append(assistant_message)

                if assistant_message.tool_calls:
                    for tool_call in assistant_message.tool_calls:
                        fname = tool_call.function.name
                        fargs = json.loads(tool_call.function.arguments)
                        if fname in available_functions:
                            tool_out = available_functions[fname](fargs)
                        else:
                            tool_out = json.dumps({"error": "Unknown function"})

                        st.session_state.messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": tool_out
                        })
                else:
                    break
        st.rerun()
"""

with open("app.py", "w", encoding="utf-8") as f:
  f.write(app_content)

print("✅ Successfully generated 'app.py' and 'requirements.txt'!")

✅ Successfully generated 'app.py' and 'requirements.txt'!


In [41]:
import getpass
import subprocess

# 1. Input your Personal Access Token securely
token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

# Define identity and repository parameters
email = "yadavamit28405@gmail.com"
username = "darkknightamkit"
repo_url = f"https://{token}@github.com/{username}/ai-coding-agent.git"

# Run commands
commands = [
    ["git", "config", "--global", "user.email", email],
    ["git", "config", "--global", "user.name", username],
    ["git", "init"],
    ["git", "add", "app.py", "requirements.txt"],
    ["git", "commit", "-m", "Initial commit: AI Coding Agent Web App"],
    ["git", "branch", "-M", "main"],
    ["git", "remote", "remove", "origin"],
    ["git", "remote", "add", "origin", repo_url],
    ["git", "push", "-u", "origin", "main"]
]

for cmd in commands:
    try:
        # Silence standard output during remote settings for security
        stdout_val = subprocess.DEVNULL if "github.com" in " ".join(cmd) else None
        subprocess.run(cmd, check=True, stdout=stdout_val, stderr=subprocess.PIPE, text=True)
    except subprocess.CalledProcessError as e:
        # Print full error details (excluding tokens in the error message for security)
        if "remove" not in cmd:
            cleaned_err = e.stderr.replace(token, "********") if token and e.stderr else "No error output"
            print(f"❌ Command failed: {' '.join(cmd)}")
            print(f"Details:\n{cleaned_err}\n")

Enter your GitHub Personal Access Token (PAT): ··········
❌ Command failed: git commit -m Initial commit: AI Coding Agent Web App
Details:
No error output

❌ Command failed: git push -u origin main
Details:
remote: Repository not found.
fatal: repository 'https://github.com/darkknightamkit/ai-coding-agent.git/' not found




In [14]:
!git --version
!git config --global user.name "darkknightamkit"
!git config --global user.email "yadavamit28405@gmail.com"
!git init
!git add .
!git status
!git commit -m "Initial commit"
!git branch -M main
!git remote add origin https://github.com/darkknightamkit/AI-agent.git
!git remote -v
!git push -u origin main

git version 2.43.0
Reinitialized existing Git repository in /content/.git/
On branch main
nothing to commit, working tree clean
On branch main
nothing to commit, working tree clean
error: remote origin already exists.
origin	https://github.com/darkknightamkit/AI-agent.git (fetch)
origin	https://github.com/darkknightamkit/AI-agent.git (push)
fatal: could not read Username for 'https://github.com': No such device or address


In [15]:
import getpass
import subprocess

# 1. Prompt for your Personal Access Token securely
token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

# 2. Define repository details
username = "darkknightamkit"
repo_name = "AI-agent"
authenticated_url = f"https://{token}@github.com/{username}/{repo_name}.git"

try:
    # 3. Clean up any existing origin
    subprocess.run(["git", "remote", "remove", "origin"], stderr=subprocess.DEVNULL)

    # 4. Add the authenticated origin
    subprocess.run(["git", "remote", "add", "origin", authenticated_url], check=True)
    print("✅ Authenticated origin remote configured.")

    # 5. Push the code securely
    print("Pushing to GitHub...")
    push_res = subprocess.run(["git", "push", "-u", "origin", "main"], capture_output=True, text=True)

    if push_res.returncode == 0:
        print("🚀 Success! Your code has been pushed to GitHub!")
    else:
        # Clean up token from any error output before displaying
        error_msg = push_res.stderr.replace(token, "********")
        print(f"❌ Push failed:\n{error_msg}")
except Exception as e:
    print(f"An error occurred: {str(e)}")

Enter your GitHub Personal Access Token (PAT): ··········
✅ Authenticated origin remote configured.
Pushing to GitHub...
🚀 Success! Your code has been pushed to GitHub!


In [ ]:
import glob
import subprocess

# Find any .ipynb files in the workspace
notebook_files = glob.glob("*.ipynb")

if not notebook_files:
    # If none in the current directory, look in the parent or typical colab path
    notebook_files = glob.glob("/content/*.ipynb")

if notebook_files:
    print(f"Found notebook files: {notebook_files}")
    try:
        # Add the notebook files to git
        for nb in notebook_files:
            subprocess.run(["git", "add", nb], check=True)
            print(f"✅ Staged: {nb}")

        # Commit the changes
        subprocess.run(["git", "commit", "-m", "Add Colab notebook file to repository"], check=True)

        # Push to GitHub using the authenticated remote we set up
        print("Pushing updated files to GitHub...")
        push_res = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)

        if push_res.returncode == 0:
            print("🚀 Success! Your Colab notebook has been successfully pushed to GitHub!")
        else:
            # Hide the token if it somehow leaks in output
            error_msg = push_res.stderr.replace(token, "********") if 'token' in globals() else push_res.stderr
            print(f"❌ Push failed:\n{error_msg}")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
else:
    print("⚠️ No .ipynb notebook file was found in the workspace directory. Make sure to download or save it inside the '/content' directory first, or you can manually upload it to GitHub.")